<a href="https://colab.research.google.com/github/ddahlgren11/MoneyMaker/blob/main/actualcapstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Add the dependencies to use tweepy for twiiter and alpaca for my stock data

# Project README: CEO Tweet and Stock Data Analysis

## Overview
This project aims to collect, classify, and analyze public tweets from prominent CEOs (Elon Musk, Tim Cook, Satya Nadella) and corresponding historical stock data for their companies (Tesla, Apple, Microsoft). The goal is to classify tweets by sentiment, tone, and type, and to prepare the data for potential future analysis relating tweet sentiment to stock market movements.

## Components

### 1. Data Collection

*   **Tweet Data:** Utilizes the `tweety-ns` library to scrape recent tweets from specified CEO Twitter accounts. For each tweet, it captures text, creation date, likes, and retweets.
*   **Stock Data:** Leverages the `alpaca-py` library to fetch historical daily stock data (Open, High, Low, Close, Volume, etc.) for the companies associated with the CEOs.

### 2. Data Preprocessing & Storage

*   **Sentiment Analysis:** Applies `TextBlob` to calculate a numerical sentiment score (polarity) for each fetched tweet.
*   **Date Handling:** Converts tweet creation dates and stock data timestamps to a consistent datetime format.
*   **Google Drive Integration:** All raw and classified data is saved to CSV files in Google Drive for persistent storage across Colab sessions.

### 3. Tweet Classification Systems

The project implements three layers of classification for tweets:

*   **Refined Sentiment:** The numerical `sentiment_score` is categorized into five distinct labels:
    *   'Very Negative' (score $\le$ -0.6)
    *   'Negative' (score $\le$ -0.2)
    *   'Neutral' (score < 0.2)
    *   'Positive' (score < 0.6)
    *   'Very Positive' (score $\ge$ 0.6)
*   **Tone/Category:** Tweets are classified into broader groups using keyword-based rules and sentiment scores:
    *   'Emotional (Joyful/Excited)', 'Emotional (Angry/Frustrated)', 'Emotional (Neutral emotion)'
    *   'Informational (Promotional/Update)', 'Informational (Personal Opinion)', 'Informational (Crisis Communication)', 'Informational (General)', 'Informational (Mixed)'
    *   'General Commentary'
*   **Detailed Tweet Type:** A comprehensive rule-based system categorizes tweets into specific types:
    *   **Engagement & Community:** 'Discussion Starter', 'Poll/Vote', 'FAQ', 'Teaser/Hype', 'Resource Share'
    *   **Daily Content:** 'Daily Chatter', 'Live-Tweet', 'Throwback', 'Tutorial/How-To'
    *   **Performance & Results:** 'Company Milestone', 'Game/Event Recap', 'Recognition/Shoutout', 'Case Study', 'Stat Blast'
    *   **Technical & Development:** 'Technical Update', 'Patch Notes / Bug Fix', 'Build in Public', 'Documentation/Manual', 'Feature Reveal'
    *   **Logistics & Organization:** 'Announcement', 'Schedule Update', 'Program/Job List'
    *   'Company-Related Discussion', 'Personal/General Commentary', 'General Commentary'

## Usage

1.  **Dependencies:** Ensure all required libraries (`tweety-ns`, `textblob`, `alpaca-py`, `pandas`) are installed using `!pip install`.
2.  **Authentication:** Set up `ALPACA_API_KEY`, `ALPACA_SECRET_KEY` as Colab secrets.
3.  **Run Cells:** Execute the notebook cells sequentially to fetch data, perform classifications, and save outputs.

## Outputs

*   `TSLA_stock_data.csv`: Historical stock data for Tesla (example).
*   `ceo_tweets_data.csv`: Raw tweet data and initial sentiment scores for all target CEOs.
*   `sample_tweet_classification_report.csv`: A CSV file containing a sample of classified tweets with refined sentiment, tone/category, and tweet type.
*   `full_tweet_classification_report.csv`: A comprehensive CSV file containing the original tweet text, refined sentiment, tone/category, and tweet type for the entire collected dataset.

## Next Steps

The classified tweet data can be used for:

*   **Further Analysis:** Exploring trends in CEO communication, sentiment over time, and distribution of tweet types.
*   **Machine Learning:** Training predictive models to forecast stock price movements based on aggregated tweet sentiment and other features.
*   **Visualization:** Creating dashboards or interactive plots to present the findings.


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# Updated dependencies for scraping and sentiment analysis
!pip install tweety-ns textblob alpaca-py pandas

Code for pulling data from twitter

In [18]:
from tweety import Twitter
from textblob import TextBlob
import pandas as pd
import time
import asyncio

# 1. Setup Tweety
app = Twitter("session")

# 2. Define targets
ceo_targets = {
    "elonmusk": "Tesla",
    "tim_cook": "Apple",
    "satyanadella": "Microsoft"
}

all_tweets = []

# 3. Asynchronous function to fetch tweets
async def fetch_tweets_async():
    for username, company in ceo_targets.items():
        try:
            print(f"Fetching tweets for {username}...")

            # In newer versions of tweety-ns, the method is get_tweets, and it's asynchronous
            # Changed pages=1 to pages=3 to get more tweets
            user_tweets = await app.get_tweets(username, pages=3)

            if user_tweets:
                for tweet in user_tweets:
                    analysis = TextBlob(tweet.text)

                    all_tweets.append({
                        'ceo': username,
                        'company': company,
                        'tweet_id': tweet.id,
                        'text': tweet.text,
                        'created_at': tweet.created_on,
                        'sentiment_score': analysis.sentiment.polarity,
                        'likes': getattr(tweet, 'likes', 0),
                        'retweets': getattr(tweet, 'retweet_counts', 0)
                    })

            # Small delay to mimic human behavior
            await asyncio.sleep(2)

        except Exception as e:
            print(f"Error fetching {username}: {e}")

# Run the asynchronous tweet fetching function
await fetch_tweets_async()

# 4. Final Dataset processing with safety check
if all_tweets:
    df_all_ceos = pd.DataFrame(all_tweets)
    # Convert date to standard format
    df_all_ceos['created_at'] = pd.to_datetime(df_all_ceos['created_at']).dt.tz_localize(None)
    print(f"\nCollection Complete! Total tweets gathered: {len(df_all_ceos)}")
    display(df_all_ceos.head())
else:
    print("\nNo tweets were collected. Check if the usernames are correct or if a login is required.")
    df_all_ceos = pd.DataFrame() # Create empty DF to prevent downstream errors

Fetching tweets for elonmusk...
Fetching tweets for tim_cook...
Fetching tweets for satyanadella...

Collection Complete! Total tweets gathered: 298


,ceo,company,tweet_id,text,created_at,sentiment_score,likes,retweets
0,elonmusk,Tesla,1519480761749016577,Next I’m buying Coca-Cola to put the cocaine b...,2022-04-28 00:56:58,0.000,4255634,588733
1,elonmusk,Tesla,1812258574049157405,https://t.co/6eOgN9UdOy,2024-07-13 22:51:28,0.000,3251245,384191
2,elonmusk,Tesla,1518623997054918657,I hope that even my worst critics remain on Tw...,2022-04-25 16:12:30,-0.300,2892883,321658
3,elonmusk,Tesla,1854026234339938528,🇺🇸🇺🇸The future is gonna be so 🔥 🇺🇸🇺🇸 https://t...,2024-11-06 05:01:15,0.000,2544815,200411
4,elonmusk,Tesla,1519495072802390016,Let’s make Twitter maximum fun!,2022-04-28 01:53:50,0.375,2365238,168140


Pulling ALPACA date

In [19]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from google.colab import userdata
from datetime import datetime

# 1. Authenticate using Colab Secrets
API_KEY = userdata.get('ALPACA_API_KEY')
SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

# Initialize the data client
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

# 2. Fetch Historical Stock Data (e.g., Tesla for Elon Musk)
# We use TimeFrame.Day to align with your "next-day prediction" goal
request_params = StockBarsRequest(
    symbol_or_symbols="TSLA",
    timeframe=TimeFrame.Day,
    start=datetime(2023, 1, 3),  # Changed start date to a valid trading day
    end=datetime(2023, 1, 4)    # Changed end date to fetch for a single trading day
)

# 3. Retrieve and Convert to a DataFrame
bars = data_client.get_stock_bars(request_params)
df_stocks = bars.df
display(df_stocks.head())

,,open,high,low,close,volume,trade_count,vwap
symbol,timestamp,,,,,,,
TSLA,2023-01-03 05:00:00+00:00,118.47,118.8,104.64,108.1,235727017.0,2540115.0,109.133537


### Saving Stock Data to CSV

I will now save the `df_stocks` DataFrame, containing the historical Tesla stock data, to a CSV file in your Google Drive. This ensures your data is persisted and can be easily accessed for further analysis.

In [20]:
from google.colab import drive
drive.mount('/content/drive')

df_stocks.to_csv('/content/drive/MyDrive/TSLA_stock_data.csv', index=False)
print("Tesla stock data saved to 'TSLA_stock_data.csv' in your Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tesla stock data saved to 'TSLA_stock_data.csv' in your Google Drive.


### Saving Tweet Sentiment Data to CSV

I will now save the `df_all_ceos` DataFrame, containing the collected tweet data and sentiment scores, to a CSV file in your Google Drive. This ensures your tweet data is also persisted and can be easily accessed for further analysis.

In [21]:
from google.colab import drive
# It's good practice to mount drive again, though it might show 'already mounted'
drive.mount('/content/drive')

df_all_ceos.to_csv('/content/drive/MyDrive/ceo_tweets_data.csv', index=False)
print("CEO tweet data saved to 'ceo_tweets_data.csv' in your Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CEO tweet data saved to 'ceo_tweets_data.csv' in your Google Drive.


### Aggregate Daily Tweet Sentiment

To align with the daily stock data, we need to aggregate the sentiment scores of tweets for each CEO on a daily basis. We'll calculate the daily average sentiment score for each CEO.

In [22]:
import pandas as pd

# Extract the date from created_at
df_all_ceos['date'] = df_all_ceos['created_at'].dt.date

# Calculate daily average sentiment for each CEO
# This now works because 'sentiment_score' is generated in Step 2
df_daily_sentiment = df_all_ceos.groupby(['ceo', 'date'])['sentiment_score'].mean().reset_index()
df_daily_sentiment = df_daily_sentiment.rename(columns={'sentiment_score': 'avg_daily_sentiment'})

print("Daily average sentiment per CEO calculated:")
display(df_daily_sentiment.head())

Daily average sentiment per CEO calculated:


,ceo,date,avg_daily_sentiment
0,elonmusk,2018-10-19,0.800
1,elonmusk,2020-03-06,-0.375
2,elonmusk,2022-01-06,0.000
3,elonmusk,2022-03-14,0.000
4,elonmusk,2022-04-07,0.000


# Task
Select 5-10 sample tweets from the `df_all_ceos` DataFrame. Then, refine the `sentiment_score` of these sample tweets into 'Very Negative', 'Negative', 'Neutral', 'Positive', and 'Very Positive' categories by defining and applying thresholds. Next, classify these sample tweets into 'Angry, joyful, excited' and 'Promotional, personal opinion, crisis communication' tone/category groups using rule-based or keyword-based methods. Subsequently, categorize the sample tweets into detailed 'Tweet Type' classifications like Discussion Starter, Poll/Vote, Daily Chatter, Company Milestone, Technical Update, or Announcement using a simplified rule-based or keyword-based approach. Finally, create a new Pandas DataFrame with the original text of the sample tweets and all the newly assigned classification columns (refined sentiment, tone/category, and tweet type), save this DataFrame as a CSV file to "/content/drive/MyDrive/sample_tweet_classification_report.csv", and display the head of this report.

## Select Sample Tweets for Classification

### Subtask:
Extract a small, manageable number of tweets (e.g., 5-10 tweets) from the `df_all_ceos` DataFrame. This sample will be used for the initial classification exercise to avoid extensive processing on the full dataset.


**Reasoning**:
To extract a small, manageable number of tweets, I will randomly sample 10 rows from the `df_all_ceos` DataFrame and store them in a new DataFrame called `df_sample_tweets`.



In [23]:
df_sample_tweets = df_all_ceos.sample(n=10, random_state=42) # Using random_state for reproducibility
print("Sample of 10 tweets selected for classification:")
display(df_sample_tweets)

Sample of 10 tweets selected for classification:


,ceo,company,tweet_id,text,created_at,sentiment_score,likes,retweets,date
159,tim_cook,Apple,1782785458881302714,Pencil us in for May 7! ✏️ #AppleEvent https:/...,2024-04-23 14:55:50,0.000000,41206,5430,2024-04-23
264,satyanadella,Microsoft,1356722820575756291,Congratulations to @JeffBezos and @ajassy on y...,2021-02-02 21:55:01,0.168182,6235,354,2021-02-02
254,satyanadella,Microsoft,973783773505564675,We lost a great one today. Stephen Hawking wil...,2018-03-14 04:51:58,0.455000,7239,1914,2018-03-14
9,elonmusk,Tesla,1812256998588662068,I fully endorse President Trump and hope for h...,2024-07-13 22:45:13,0.000000,2232867,340589,2024-07-13
77,elonmusk,Tesla,1520017094007476224,"The far left hates everyone, themselves included!",2022-04-29 12:28:10,0.050000,1052019,104870,2022-04-29
234,satyanadella,Microsoft,1892242897216450621,"Read more about our discovery, and why it matt...",2025-02-19 16:00:37,0.500000,11845,1972,2025-02-19
93,elonmusk,Tesla,1851971913586786676,https://t.co/D3xwF6cWBr,2024-10-31 12:58:06,0.000000,1019373,79896,2024-10-31
109,tim_cook,Apple,1305823678584827905,Good morning! T-minus 6 hours! #AppleEvent,2020-09-15 11:00:00,1.000000,85842,9414,2020-09-15
5,elonmusk,Tesla,1518677066325053441,🚀💫♥️ Yesss!!! ♥️💫🚀 https://t.co/0T9HzUHuh6,2022-04-25 19:43:22,0.000000,2336354,302855,2022-04-25
172,tim_cook,Apple,1900381371027083501,Happy Holi to all those celebrating! \n\nMay i...,2025-03-14 03:00:00,0.716667,37576,2046,2025-03-14


**Reasoning**:
Now that the sample tweets have been selected, the next step is to refine the `sentiment_score` into categorical sentiment labels: 'Very Negative', 'Negative', 'Neutral', 'Positive', and 'Very Positive'. This involves defining thresholds for the existing numerical sentiment scores and applying a function to classify each tweet.



In [24]:
def classify_sentiment(score):
    if score <= -0.6:
        return 'Very Negative'
    elif score <= -0.2:
        return 'Negative'
    elif score < 0.2:
        return 'Neutral'
    elif score < 0.6:
        return 'Positive'
    else:
        return 'Very Positive'

# Apply the classification function to the sentiment_score column of df_sample_tweets
df_sample_tweets['refined_sentiment'] = df_sample_tweets['sentiment_score'].apply(classify_sentiment)

print("Refined sentiment categories added to df_sample_tweets:")
display(df_sample_tweets[['text', 'sentiment_score', 'refined_sentiment']].head())

Refined sentiment categories added to df_sample_tweets:


,text,sentiment_score,refined_sentiment
159,Pencil us in for May 7! ✏️ #AppleEvent https:/...,0.000000,Neutral
264,Congratulations to @JeffBezos and @ajassy on y...,0.168182,Neutral
254,We lost a great one today. Stephen Hawking wil...,0.455000,Positive
9,I fully endorse President Trump and hope for h...,0.000000,Neutral
77,"The far left hates everyone, themselves included!",0.050000,Neutral


## Classify Tweets by Tone/Category

### Subtask:
Classify the sample tweets into broader 'Tone/Category' groups. This classification aims to understand the overall sentiment and intent behind the tweets, such as 'Emotional (Angry, Joyful, Excited)' or 'Informational (Promotional, Personal Opinion, Crisis Communication)'.

### Instructions:
1. Define functions or rules to classify tweets into 'Emotional' (e.g., Angry, Joyful, Excited) or 'Informational' (e.g., Promotional, Personal Opinion, Crisis Communication).
2. Apply these rules to the `df_sample_tweets` DataFrame and add a new column for 'Tone/Category'.

**Reasoning**:
To classify tweets into tone/category groups, I will define a function that checks for keywords associated with 'Emotional' and 'Informational' themes within the tweet text. This function will then be applied to the 'text' column of the `df_sample_tweets` DataFrame to populate a new 'tone_category' column.



In [25]:
def classify_tone_category(text, sentiment_score):
    text_lower = str(text).lower()

    # Keywords for Emotional categories (Angry, Joyful, Excited)
    emotional_keywords = [
        "excited", "happy", "joyful", "amazing", "great", "love",
        "angry", "frustrated", "disappointed", "sad", "horrible", "bad"
    ]

    # Keywords for Informational categories (Promotional, Personal Opinion, Crisis Communication)
    informational_keywords = [
        "announcement", "new feature", "launch", "event", "update", "release",
        "opinion", "think", "believe", "my take",
        "crisis", "issue", "statement", "apology", "urgent"
    ]

    is_emotional = any(keyword in text_lower for keyword in emotional_keywords)
    is_informational = any(keyword in text_lower for keyword in informational_keywords)

    if is_emotional and is_informational:
        # If both, prioritize based on sentiment or specific dominant keywords (for simplicity, default to informational if both present with neutral sentiment)
        if sentiment_score > 0.2: # Leaning positive sentiment implies joy/excitement
            return 'Emotional (Joyful/Excited)'
        elif sentiment_score < -0.2: # Leaning negative sentiment implies angry/frustrated
            return 'Emotional (Angry/Frustrated)'
        else:
            return 'Informational (Mixed)' # If sentiment is neutral, likely an informational tweet with some emotional undertones
    elif is_emotional:
        if sentiment_score > 0: # Positive emotional
            return 'Emotional (Joyful/Excited)'
        elif sentiment_score < 0: # Negative emotional
            return 'Emotional (Angry/Frustrated)'
        else:
            return 'Emotional (Neutral emotion)'
    elif is_informational:
        # Further classify informational
        if any(kw in text_lower for kw in ["announcement", "new feature", "launch", "event", "update", "release"]):
            return 'Informational (Promotional/Update)'
        elif any(kw in text_lower for kw in ["opinion", "think", "believe", "my take"]):
            return 'Informational (Personal Opinion)'
        elif any(kw in text_lower for kw in ["crisis", "issue", "statement", "apology", "urgent"]):
            return 'Informational (Crisis Communication)'
        else:
            return 'Informational (General)'
    else:
        # Default to personal opinion if no specific keywords are found but the tweet is by a CEO
        # Or if it's general commentary
        if "i" in text_lower or "my" in text_lower: # Simple heuristic for personal opinion
             return 'Informational (Personal Opinion)'
        else:
            return 'General Commentary'

# Apply the function to create the 'tone_category' column
df_sample_tweets['tone_category'] = df_sample_tweets.apply(lambda row: classify_tone_category(row['text'], row['sentiment_score']), axis=1)

print("Tone/Category classification added to df_sample_tweets:")
display(df_sample_tweets[['text', 'refined_sentiment', 'tone_category']].head())

Tone/Category classification added to df_sample_tweets:


,text,refined_sentiment,tone_category
159,Pencil us in for May 7! ✏️ #AppleEvent https:/...,Neutral,Informational (Promotional/Update)
264,Congratulations to @JeffBezos and @ajassy on y...,Neutral,Informational (Personal Opinion)
254,We lost a great one today. Stephen Hawking wil...,Positive,Emotional (Joyful/Excited)
9,I fully endorse President Trump and hope for h...,Neutral,Informational (Personal Opinion)
77,"The far left hates everyone, themselves included!",Neutral,Informational (Personal Opinion)


**Reasoning**:
To further categorize the sample tweets into detailed 'Tweet Type' classifications, I will define a function that uses a rule-based or keyword-based approach to identify types such as Discussion Starter, Poll/Vote, Daily Chatter, Company Milestone, Technical Update, or Announcement. This function will be applied to the `df_sample_tweets` DataFrame to add a new 'tweet_type' column.



In [26]:
# Send the merged DataFrame to the API
send_to_api(df_merged, 'merged')

print("Merged data sent to API.")

NameError: name 'send_to_api' is not defined

In [ ]:
def classify_tweet_type(text, ceo_name):
    text_lower = str(text).lower()

    # Keywords for different tweet types
    if "poll" in text_lower or "vote" in text_lower or "which do you prefer" in text_lower:
        return 'Poll/Vote'
    elif "what do you think" in text_lower or "thoughts on" in text_lower or "discuss" in text_lower or "question" in text_lower:
        return 'Discussion Starter'
    elif "announcement" in text_lower or "launch" in text_lower or "event" in text_lower or "release" in text_lower or "update" in text_lower:
        return 'Company Milestone/Announcement'
    elif "technical" in text_lower or "spec" in text_lower or "code" in text_lower or "software" in text_lower or "hardware" in text_lower:
        return 'Technical Update'
    elif "good morning" in text_lower or "hello twitter" in text_lower or "how's everyone" in text_lower or "daily thought" in text_lower:
        return 'Daily Chatter'
    else:
        # Fallback to more general types based on CEO activity
        if ceo_name in ['elonmusk', 'tim_cook', 'satyanadella']:
            if any(keyword in text_lower for keyword in ['tesla', 'apple', 'microsoft', 'company', 'we', 'our']):
                return 'Company-Related Discussion'
            else:
                return 'Personal/General Commentary'
        return 'General Commentary'

# Apply the function to create the 'tweet_type' column
df_sample_tweets['tweet_type'] = df_sample_tweets.apply(lambda row: classify_tweet_type(row['text'], row['ceo']), axis=1)

print("Tweet Type classification added to df_sample_tweets:")
display(df_sample_tweets[['text', 'refined_sentiment', 'tone_category', 'tweet_type']].head())

**Reasoning**:
Now that all classifications (refined sentiment, tone/category, and tweet type) have been added to `df_sample_tweets`, I will create a new DataFrame that includes the original tweet text and these new classification columns. This new DataFrame will then be saved as a CSV file to Google Drive and its head displayed as the final step of the subtask.



In [ ]:
classification_report_df = df_sample_tweets[['text', 'refined_sentiment', 'tone_category', 'tweet_type']]

# Send the DataFrame to the API
send_to_api(classification_report_df, 'sample_tweet_classification_report')

print("Classification Report DataFrame created and sent to API:")
display(classification_report_df.head())

## Summary:

### Data Analysis Key Findings

*   **Sample Selection:** A sample of 10 tweets was randomly selected from the `df_all_ceos` DataFrame for classification.
*   **Refined Sentiment Classification:** The numerical `sentiment_score` was refined into five categories: 'Very Negative' (score $\le$ -0.6), 'Negative' (score $\le$ -0.2), 'Neutral' (score $<$ 0.2), 'Positive' (score $<$ 0.6), and 'Very Positive' (score $\ge$ 0.6).
*   **Tone/Category Classification:** Tweets were classified into 'Tone/Category' groups using a rule-based method incorporating keywords and sentiment scores. Categories included 'Emotional (Joyful/Excited)', 'Emotional (Angry/Frustrated)', 'Informational (Promotional/Update)', 'Informational (Personal Opinion)', 'Informational (Crisis Communication)', and 'General Commentary'.
*   **Detailed Tweet Type Classification:** A simplified rule-based/keyword-based approach was used to categorize tweets into specific 'Tweet Types' such as 'Poll/Vote', 'Discussion Starter', 'Company Milestone/Announcement', 'Technical Update', 'Daily Chatter', 'Company-Related Discussion', and 'Personal/General Commentary'.
*   **Classification Report Generation:** A new Pandas DataFrame was created, incorporating the original tweet text along with the newly assigned 'refined\_sentiment', 'tone\_category', and 'tweet\_type' classifications.
*   **Output:** The final classification report was saved as a CSV file named `sample_tweet_classification_report.csv` to `/content/drive/MyDrive/`.

### Insights or Next Steps

*   The current keyword-based classification for tone and tweet type can be enhanced by incorporating natural language processing (NLP) models for more nuanced and accurate categorization, especially for tweets with ambiguous language or complex contexts.
*   The generated classification report can serve as a valuable dataset for training and evaluating machine learning models to automate tweet classification on a larger scale, moving beyond rule-based methods.


# Task
Apply the `classify_sentiment` function to the `sentiment_score` column of the `df_all_ceos` DataFrame to create a new column named 'refined_sentiment'.

## Apply Refined Sentiment to Full Dataset

### Subtask:
Apply the `classify_sentiment` function to the `sentiment_score` column of the entire `df_all_ceos` DataFrame to create the 'refined_sentiment' column.


**Reasoning**:
To fulfill the subtask, I will apply the `classify_sentiment` function, which was previously defined, to the `sentiment_score` column of the `df_all_ceos` DataFrame. This will create a new 'refined_sentiment' column, and then I will display the head of the DataFrame to show the results.



In [ ]:
df_all_ceos['refined_sentiment'] = df_all_ceos['sentiment_score'].apply(classify_sentiment)

print("Refined sentiment categories applied to the full df_all_ceos DataFrame:")
display(df_all_ceos[['text', 'sentiment_score', 'refined_sentiment']].head())

## Apply Tone/Category to Full Dataset

### Subtask:
Apply the `classify_tone_category` function to the `text` and `sentiment_score` columns of the entire `df_all_ceos` DataFrame to create the 'tone_category' column.


**Reasoning**:
To apply the `classify_tone_category` function to the full `df_all_ceos` DataFrame and create the 'tone_category' column, I will use the `apply` method, passing the 'text' and 'sentiment_score' columns as arguments, and then display the head of the DataFrame to verify the new column.



In [ ]:
df_all_ceos['tone_category'] = df_all_ceos.apply(lambda row: classify_tone_category(row['text'], row['sentiment_score']), axis=1)

print("Tone/Category classification applied to the full df_all_ceos DataFrame:")
display(df_all_ceos[['text', 'refined_sentiment', 'tone_category']].head())

## Apply Tweet Type to Full Dataset

### Subtask:
Apply the `classify_tweet_type` function to the `text` and `ceo` columns of the entire `df_all_ceos` DataFrame to create the 'tweet_type' column.


**Reasoning**:
To apply the `classify_tweet_type` function to the full `df_all_ceos` DataFrame and create the 'tweet_type' column, I will use the `apply` method, passing the 'text' and 'ceo' columns as arguments, and then display the head of the DataFrame to verify the new column.



In [ ]:
df_all_ceos['tweet_type'] = df_all_ceos.apply(lambda row: classify_tweet_type(row['text'], row['ceo']), axis=1)

print("Tweet Type classification applied to the full df_all_ceos DataFrame:")
display(df_all_ceos[['text', 'refined_sentiment', 'tone_category', 'tweet_type']].head())

## Save Full Classified Data to CSV

### Subtask:
Create a new Pandas DataFrame with the original text and all the newly assigned classification columns from the full `df_all_ceos` DataFrame. Save this DataFrame as a CSV file to '/content/drive/MyDrive/full_tweet_classification_report.csv'.


**Reasoning**:
To complete the subtask, I will create a new DataFrame `full_classification_report_df` with the specified columns from `df_all_ceos`, save it as a CSV file to Google Drive, print a confirmation message, and then display the head of the new DataFrame.



In [ ]:
full_classification_report_df = df_all_ceos[['text', 'refined_sentiment', 'tone_category', 'tweet_type']]

# Send the DataFrame to the API
send_to_api(full_classification_report_df, 'full_tweet_classification_report')

print("Full Classification Report DataFrame created and sent to API:")
display(full_classification_report_df.head())

In [ ]:
import pandas as pd

# 1. Prepare df_stocks: Extract date and map symbol to company name
# Reset index to make 'timestamp' a regular column
df_stocks = df_stocks.reset_index()

df_stocks['date'] = pd.to_datetime(df_stocks['timestamp']).dt.date

# Create a mapping for symbols to company names for merging
symbol_to_company = {
    'TSLA': 'Tesla',
    'AAPL': 'Apple',
    'MSFT': 'Microsoft'
}
# Note: Currently df_stocks only contains TSLA data.
# If more stock symbols are added, this mapping will be used.

df_stocks['company'] = df_stocks['symbol'].map(symbol_to_company)

print("Prepared df_stocks for merging:")
display(df_stocks.head())

In [ ]:
# 2. Merge df_all_ceos and df_stocks on 'date' and 'company'
# Using an inner merge to only keep rows where both tweet and stock data exist for a given date and company.
df_merged = pd.merge(
    df_all_ceos,
    df_stocks[['date', 'company', 'open', 'high', 'low', 'close', 'volume']], # Select relevant stock columns
    on=['date', 'company'],
    how='inner'
)

print("Merged DataFrame (df_merged) created:")
display(df_merged.head())

In [ ]:
import requests

def send_to_api(dataframe, data_name):
    """
    Sends a pandas DataFrame to the MoneyMaker API on Render.
    Args:
        dataframe (pd.DataFrame): The DataFrame to send.
        data_name (str): Should be 'tweets', 'stocks', or 'merged'.
    """
    # 1. Map the data_name to your actual Render endpoints
    base_url = "https://moneymaker-1vt4.onrender.com/ingest"
    endpoint_map = {
        "tweets": f"{base_url}/tweets",
        "stocks": f"{base_url}/stocks",
        "merged": f"{base_url}/merged"
    }

    api_endpoint = endpoint_map.get(data_name)
    if not api_endpoint:
        print(f"❌ Error: data_name '{data_name}' not recognized.")
        return

    # 2. Convert DataFrame to a list of dictionaries (JSON format)
    # This matches the List[Schema] format in your main.py
    json_payload = dataframe.to_dict(orient='records')

    print(f"🚀 Sending {len(json_payload)} rows to {api_endpoint}...")

    # 3. Actual API Call
    try:
        response = requests.post(api_endpoint, json=json_payload)

        if response.status_code == 200:
            print(f"✅ Success! {data_name} data saved to Neon. Response: {response.json()}")
        else:
            print(f"❌ API Error {response.status_code}: {response.text}")

    except requests.exceptions.RequestException as e:
        print(f"⚠️ Connection failed: {e}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

send_to_api(df_stocks, 'TSLA_stock_data')
print("Tesla stock data sent to API.")

In [ ]:
from google.colab import drive
# It's good practice to mount drive again, though it might show 'already mounted'
drive.mount('/content/drive')

send_to_api(df_all_ceos, 'ceo_tweets_data')
print("CEO tweet data sent to API.")

## Final Task

### Subtask:
Confirm that the classifications have been applied to the larger dataset and that the full classification report has been saved to your Google Drive.


## Summary:

### Q&A
Yes, the classifications (`refined_sentiment`, `tone_category`, and `tweet_type`) have been successfully applied to the larger `df_all_ceos` dataset. The full classification report has also been saved to your Google Drive as `full_tweet_classification_report.csv` in the `/content/drive/MyDrive/` directory.

### Data Analysis Key Findings

*   A new column, `refined_sentiment`, was successfully added to the `df_all_ceos` DataFrame, classifying sentiment scores into categories such as 'Neutral' (for a score of $0.000$), 'Negative' (for $-0.300$), and 'Positive' (for $0.375$).
*   The `df_all_ceos` DataFrame was further enhanced with a 'tone\_category' column, classifying entries into categories like "Informational (Personal Opinion)" and "General Commentary" based on text and sentiment.
*   A 'tweet\_type' column was created in `df_all_ceos`, categorizing tweets, with examples such as 'Personal/General Commentary' observed.
*   A new Pandas DataFrame, `full_classification_report_df`, was successfully created, containing the original 'text' along with the newly generated 'refined\_sentiment', 'tone\_category', and 'tweet\_type' columns.
*   This `full_classification_report_df` was successfully saved as a CSV file named `full_tweet_classification_report.csv` to the specified Google Drive location (`/content/drive/MyDrive/`).

### Insights or Next Steps
*   The comprehensive classification process has successfully enriched the `df_all_ceos` dataset, providing multi-faceted insights into sentiment, tone, and tweet type, which are now ready for deeper analysis or visualization.
*   The saved classification report can be easily shared or used as input for subsequent machine learning tasks or advanced data modeling.
